# Evaluación final — Reseñas de vinos

**Módulo 06 — Fundamentos de redes neuronales.** Resolución del cuestionario final sobre el dataset `winemag-data-130k-v2.csv`. El enunciado completo está en [`enunciado.md`](enunciado.md).

## Cómo usar esta notebook

El dataset **no está versionado** en el repositorio (los archivos de datos pesados están en el `.gitignore`). Antes de ejecutar, descargalo del repositorio de GitHub del curso y dejalo en:

```
practica/evaluacion-final/datasets/winemag-data-130k-v2.csv
```

## Plan de trabajo

1. Carga y exploración inicial.
2. Copia del original y eliminación de columnas.
3. `data_eliminados`: descarte de nulos y codificación de etiquetas.
4. `data_imputados`: imputación de `points` y `price`, descarte del resto de nulos y codificación.
5. División 70/30 con semilla 17 en ambos conjuntos.
6. Modelos y evaluación, a medida que el cuestionario los pida.

## 1. Librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

RANDOM_STATE = 17   # semilla pedida por el enunciado
TEST_SIZE = 0.3     # division 70/30

pd.set_option('display.max_columns', None)

## 2. Carga del dataset

Se carga el archivo original y se lo deja intacto en `datos`. Todo el trabajo posterior se hace sobre copias.

In [2]:
RUTA = 'datasets/winemag-data-130k-v2.csv'

datos = pd.read_csv(RUTA)
print(f'Filas: {datos.shape[0]:,} | Columnas: {datos.shape[1]}')
datos.head()

Filas: 129,971 | Columnas: 14


,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


### Exploración inicial

Antes de tocar nada conviene ver los tipos de dato y el panorama de valores ausentes: de eso dependen las dos estrategias que compara el cuestionario.

In [3]:
datos.info()

<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  str    
 2   description            129971 non-null  str    
 3   designation            92506 non-null   str    
 4   points                 129971 non-null  int64  
 5   price                  120975 non-null  float64
 6   province               129908 non-null  str    
 7   region_1               108724 non-null  str    
 8   region_2               50511 non-null   str    
 9   taster_name            103727 non-null  str    
 10  taster_twitter_handle  98758 non-null   str    
 11  title                  129971 non-null  str    
 12  variety                129970 non-null  str    
 13  winery                 129971 non-null  str    
dtypes: float64(1), int64(2), str(11)
memory usage: 

In [4]:
nulos = datos.isnull().sum().sort_values(ascending=False)
pd.DataFrame({
    'nulos': nulos,
    'porcentaje': (nulos / len(datos) * 100).round(2)
})

,nulos,porcentaje
region_2,79460,61.14
designation,37465,28.83
taster_twitter_handle,31213,24.02
taster_name,26244,20.19
region_1,21247,16.35
price,8996,6.92
country,63,0.05
province,63,0.05
variety,1,0.00
Unnamed: 0,0,0.00


## 3. Copia y eliminación de columnas

Según el enunciado: copia profunda del original y descarte de `region_2`, `taster_twitter_handle`, `designation` y `Unnamed: 0`.

`copy(deep=True)` crea una copia independiente: modificar `datos_copy` no altera `datos`.

In [5]:
datos_copy = datos.copy(deep = True)

columnas_a_eliminar = ['region_2', 'taster_twitter_handle', 'designation', 'Unnamed: 0']
datos_copy = datos_copy.drop(columns=columnas_a_eliminar)

print('Columnas restantes:', list(datos_copy.columns))
print(f'Dimensiones: {datos_copy.shape}')

Columnas restantes: ['country', 'description', 'points', 'price', 'province', 'region_1', 'taster_name', 'title', 'variety', 'winery']
Dimensiones: (129971, 10)


## 4. `data_eliminados`

Estrategia de **descarte**: se eliminan todas las filas que tengan algún valor ausente y después se codifican las variables categóricas.

El orden importa: primero se descartan los nulos, después se codifica. Codificar antes obligaría a decidir qué hacer con los `NaN` dentro del encoder.

In [6]:
data_eliminados = datos_copy.dropna(axis=0)

print(f'Antes : {datos_copy.shape[0]:,} filas')
print(f'Despues: {data_eliminados.shape[0]:,} filas')
print(f'Se descarto el {(1 - len(data_eliminados)/len(datos_copy))*100:.2f}% de las filas')

Antes : 129,971 filas
Despues: 77,267 filas
Se descarto el 40.55% de las filas


> **Atención con este descarte.** No se pierden solo filas: se pierden **clases enteras**. La celda siguiente lo verifica.

In [7]:
paises_orig = datos_copy['country'].nunique()
paises_post = data_eliminados['country'].nunique()

print(f'Paises en el original      : {paises_orig}')
print(f'Paises tras eliminar nulos : {paises_post}')
print(f'\nDistribucion resultante:')
print(data_eliminados['country'].value_counts().to_string())

print(f"\nPaises con region_1 no nula: {datos_copy[datos_copy['region_1'].notna()]['country'].nunique()}")

Paises en el original      : 43
Paises tras eliminar nulos : 7

Distribucion resultante:
country
US           37255
France       17457
Italy        10096
Spain         6501
Argentina     3700
Australia     2005
Canada         253

Paises con region_1 no nula: 7


El culpable es **`region_1`**: solo está poblada para 7 países, así que al descartar las filas con algún ausente desaparecen los otros 36. El conjunto `data_eliminados` no es una muestra representativa del original, es el subconjunto de países que tienen región cargada.

Además, las 7 clases que quedan están **muy desbalanceadas**: US aporta unas 37.000 filas y Canadá apenas 253. Al evaluar la clasificación de `country`, la exactitud sola va a estar dominada por US — conviene mirar la matriz de confusión y el F1 por clase (cap. 8 del manual).

### Codificación de etiquetas

La **codificación de etiquetas** (`LabelEncoder`) asigna un entero a cada categoría. Se aplica a todas las columnas de tipo objeto.

> Nota: esta codificación introduce un orden numérico que no existe entre las categorías (que Argentina sea 1 y Australia 2 no significa que Australia sea "mayor"). Los modelos basados en árboles lo toleran bien; los modelos lineales y las redes neuronales pueden interpretarlo como una magnitud. El enunciado pide explícitamente esta codificación, así que se usa esta.

In [8]:
data_eliminados = data_eliminados.copy()

# Se codifican solo las columnas categoricas que se van a usar como feature o target.
# 'description' y 'title' son texto libre de altisima cardinalidad (casi un valor
# distinto por fila): codificarlas con etiquetas no aporta nada y solo gasta tiempo.
columnas_categoricas = ['country', 'province', 'region_1', 'taster_name', 'variety', 'winery']

encoders_eliminados = {}
for col in columnas_categoricas:
    le = LabelEncoder()
    data_eliminados[col] = le.fit_transform(data_eliminados[col])
    encoders_eliminados[col] = le

print('Codificadas:', columnas_categoricas)
data_eliminados.head()

Codificadas: ['country', 'province', 'region_1', 'taster_name', 'variety', 'winery']


,country,description,points,price,province,region_1,taster_name,title,variety,winery
2,6,"Tart and snappy, the flavors of lime flesh and...",87,14.0,43,1079,12,Rainstorm 2013 Pinot Gris (Willamette Valley),288,9309
3,6,"Pineapple rind, lemon pith and orange blossom ...",87,13.0,30,491,0,St. Julian 2013 Reserve Late Harvest Riesling ...,316,10215
4,6,"Much like the regular bottling from 2012, this...",87,65.0,43,1079,12,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,292,10339
5,5,Blackberry and raspberry aromas show a typical...,87,15.0,39,663,11,Tandem 2011 Ars In Vitro Tempranillo-Merlot (N...,394,10396
6,4,"Here's a bright, informal red that opens with ...",87,16.0,49,1066,8,Terre di Giurfo 2013 Belsito Frappato (Vittoria),113,10603


### Separación de features y target

Target: `country`. Es un problema de **clasificación multiclase**.

In [9]:
features = ['price', 'points', 'province', 'region_1','taster_name','variety', 'winery']
target = 'country'

X_eliminados = data_eliminados[features]
y_eliminados = data_eliminados[target]

print(f'X_eliminados: {X_eliminados.shape}')
print(f'y_eliminados: {y_eliminados.shape} | clases distintas: {y_eliminados.nunique()}')

X_eliminados: (77267, 7)
y_eliminados: (77267,) | clases distintas: 7


## 5. `data_imputados`

Estrategia de **imputación**: se completan `points` con su mediana y `price` con su media, y recién después se descartan las filas que sigan teniendo ausentes en otras columnas.

- **Mediana** para `points`: es robusta a valores extremos.
- **Media** para `price`: es la que pide el enunciado.

> **Observación metodológica.** El target de este conjunto es `price`, y el enunciado pide imputarlo con la media antes de separarlo. Imputar la variable objetivo introduce filas cuyo valor real es desconocido y que el modelo aprende como si fueran observaciones válidas — se le enseña a predecir la media. Es una decisión discutible, pero es lo que pide la consigna, así que se sigue al pie de la letra. Vale tenerlo presente al interpretar las métricas.

In [10]:
data_imputados = datos_copy.copy(deep = True)

mediana_points = data_imputados['points'].median()
media_price = data_imputados['price'].mean()

print(f'Mediana de points: {mediana_points}')
print(f'Media de price   : {media_price:.4f}')

data_imputados['points'] = data_imputados['points'].fillna(mediana_points)
data_imputados['price'] = data_imputados['price'].fillna(media_price)

print(f"\nNulos restantes en points: {data_imputados['points'].isnull().sum()}")
print(f"Nulos restantes en price : {data_imputados['price'].isnull().sum()}")

Mediana de points: 88.0
Media de price   : 35.3634

Nulos restantes en points: 0
Nulos restantes en price : 0


Ahora sí, se descartan las filas que sigan teniendo ausentes en el resto de las columnas.

In [11]:
antes = data_imputados.shape[0]
data_imputados = data_imputados.dropna(axis=0)

print(f'Antes : {antes:,} filas')
print(f'Despues: {data_imputados.shape[0]:,} filas')
print(f'\nComparacion con la estrategia de descarte:')
print(f'  data_eliminados: {data_eliminados.shape[0]:,} filas')
print(f'  data_imputados : {data_imputados.shape[0]:,} filas')

Antes : 129,971 filas
Despues: 82,847 filas

Comparacion con la estrategia de descarte:
  data_eliminados: 77,267 filas
  data_imputados : 82,847 filas


### Codificación de etiquetas

In [12]:
data_imputados = data_imputados.copy()

columnas_categoricas_imp = ['country', 'province', 'region_1', 'taster_name', 'variety', 'winery']

encoders_imputados = {}
for col in columnas_categoricas_imp:
    le = LabelEncoder()
    data_imputados[col] = le.fit_transform(data_imputados[col])
    encoders_imputados[col] = le

print('Codificadas:', columnas_categoricas_imp)
data_imputados.head()

Codificadas: ['country', 'province', 'region_1', 'taster_name', 'variety', 'winery']


,country,description,points,price,province,region_1,taster_name,title,variety,winery
0,4,"Aromas include tropical fruit, broom, brimston...",87,35.363389,49,386,8,Nicosia 2013 Vulkà Bianco (Etna),465,8938
2,6,"Tart and snappy, the flavors of lime flesh and...",87,14.000000,43,1096,12,Rainstorm 2013 Pinot Gris (Willamette Valley),292,9860
3,6,"Pineapple rind, lemon pith and orange blossom ...",87,13.000000,30,499,0,St. Julian 2013 Reserve Late Harvest Riesling ...,320,10783
4,6,"Much like the regular bottling from 2012, this...",87,65.000000,43,1096,12,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,296,10909
5,5,Blackberry and raspberry aromas show a typical...,87,15.000000,39,674,11,Tandem 2011 Ars In Vitro Tempranillo-Merlot (N...,400,10968


### Separación de features y target

Target: `price`. Es un problema de **regresión**.

In [13]:
features = ['country', 'points', 'province', 'region_1','taster_name','variety', 'winery']
target = 'price'

X_imputados = data_imputados[features]
y_imputados = data_imputados[target]

print(f'X_imputados: {X_imputados.shape}')
print(f'y_imputados: {y_imputados.shape}')
y_imputados.describe()

X_imputados: (82847, 7)
y_imputados: (82847,)


count    82847.000000
mean        36.808064
std         43.072425
min          4.000000
25%         18.000000
50%         29.000000
75%         42.000000
max       3300.000000
Name: price, dtype: float64

## 6. División de los conjuntos

70/30 con `random_state=17` en ambos casos, tal como pide el enunciado.

In [14]:
# Conjunto de descarte -> clasificacion de country
X_train_el, X_test_el, y_train_el, y_test_el = train_test_split(
    X_eliminados, y_eliminados, test_size=TEST_SIZE, random_state=RANDOM_STATE)

# Conjunto imputado -> regresion de price
X_train_im, X_test_im, y_train_im, y_test_im = train_test_split(
    X_imputados, y_imputados, test_size=TEST_SIZE, random_state=RANDOM_STATE)

print('ELIMINADOS (clasificacion de country)')
print(f'  train: {X_train_el.shape[0]:,} | test: {X_test_el.shape[0]:,}')
print('IMPUTADOS (regresion de price)')
print(f'  train: {X_train_im.shape[0]:,} | test: {X_test_im.shape[0]:,}')

ELIMINADOS (clasificacion de country)
  train: 54,086 | test: 23,181
IMPUTADOS (regresion de price)
  train: 57,992 | test: 24,855


## 7. Punto de partida para los modelos

Todo lo anterior deja listo:

| Variable | Contenido |
|---|---|
| `X_train_el`, `X_test_el`, `y_train_el`, `y_test_el` | conjunto de **descarte** — clasificación de `country` |
| `X_train_im`, `X_test_im`, `y_train_im`, `y_test_im` | conjunto **imputado** — regresión de `price` |
| `encoders_eliminados`, `encoders_imputados` | los `LabelEncoder` de cada columna, para revertir la codificación con `inverse_transform` |

Recordatorios para cuando se entrenen los modelos:

- Mantener `random_state=17` en cada estimador, o los resultados no van a ser reproducibles.
- Si se usan redes neuronales o cualquier modelo basado en distancias, **escalar** los datos: ajustar el `StandardScaler` solo con train, o mejor, meterlo en un `Pipeline`.
- En clasificación de `country` las clases están muy desbalanceadas: la exactitud sola puede engañar, conviene mirar también la matriz de confusión y el F1.

## 8. Resumen de números clave

Los valores que salen de la preparación, para tener a mano al responder el cuestionario.

In [15]:
resumen = pd.DataFrame([
    ['Filas del dataset original',            f'{datos.shape[0]:,}'],
    ['Columnas tras el descarte',             f'{datos_copy.shape[1]}'],
    ['Mediana de points (imputacion)',        f'{mediana_points}'],
    ['Media de price (imputacion)',           f'{media_price:.4f}'],
    ['data_eliminados: filas',                f'{data_eliminados.shape[0]:,}'],
    ['data_eliminados: clases de country',    f'{y_eliminados.nunique()}'],
    ['data_imputados: filas',                 f'{data_imputados.shape[0]:,}'],
    ['Train eliminados / Test eliminados',    f'{X_train_el.shape[0]:,} / {X_test_el.shape[0]:,}'],
    ['Train imputados / Test imputados',      f'{X_train_im.shape[0]:,} / {X_test_im.shape[0]:,}'],
], columns=['Concepto', 'Valor'])

resumen

,Concepto,Valor
0,Filas del dataset original,"129,971"
1,Columnas tras el descarte,10
2,Mediana de points (imputacion),88.0
3,Media de price (imputacion),35.3634
4,data_eliminados: filas,"77,267"
5,data_eliminados: clases de country,7
6,data_imputados: filas,"82,847"
7,Train eliminados / Test eliminados,"54,086 / 23,181"
8,Train imputados / Test imputados,"57,992 / 24,855"


---

## 9. Consignas del cuestionario

Cada consigna queda con su enunciado textual, el código que la resuelve y la respuesta.

In [16]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

### Consigna 1 — MLP para clasificar el país

> Considerando el DataFrame `data_eliminados`: crea un MLP para poder clasificar los países de las referencias de vinos, en base a las features mencionadas. Para el MLP establece el valor de semilla = 17, con 2 capas ocultas, la primera con 6 neuronas y la segunda con 12 y establece un máximo de 100 iteraciones. Luego de haber entrenado el modelo y evaluarlo, el valor de Accuracy es:

Traducido a parámetros: `hidden_layer_sizes=(6, 12)`, `max_iter=100`, `random_state=17`. La consigna no menciona escalado, así que se entrena directamente sobre `X_train_el`.

In [17]:
mlp_paises = MLPClassifier(hidden_layer_sizes=(6, 12), max_iter=100, random_state=17)
mlp_paises.fit(X_train_el, y_train_el)

y_pred_el = mlp_paises.predict(X_test_el)
accuracy_el = accuracy_score(y_test_el, y_pred_el)

print(f'Accuracy: {accuracy_el:.4f}   ->   {accuracy_el*100:.2f}%')
print(f'\nn_iter_: {mlp_paises.n_iter_} de 100 | loss final: {mlp_paises.loss_:.4f}')

Accuracy: 0.5856   ->   58.56%

n_iter_: 73 de 100 | loss final: 1.1987


**Respuesta: Accuracy ≈ 0,5856 (58,56%).**

#### Qué hay detrás de ese número

Tres cosas que conviene mirar antes de darlo por bueno.

In [18]:
# 1. El baseline: predecir siempre la clase mayoritaria
baseline = y_test_el.value_counts(normalize=True).max()
print(f'Predecir siempre la clase mayoritaria (US) daria: {baseline*100:.2f}%')
print(f'El modelo logra                                 : {accuracy_el*100:.2f}%')
print(f'Mejora sobre el baseline                        : {(accuracy_el-baseline)*100:.2f} puntos')

# 2. F1 macro: promedia el F1 de cada clase sin ponderar por frecuencia
print(f'\nF1 macro: {f1_score(y_test_el, y_pred_el, average="macro"):.4f}')
print(f'Clases distintas predichas: {pd.Series(y_pred_el).nunique()} de {y_test_el.nunique()}')

Predecir siempre la clase mayoritaria (US) daria: 48.10%
El modelo logra                                 : 58.56%
Mejora sobre el baseline                        : 10.46 puntos

F1 macro: 0.2491
Clases distintas predichas: 6 de 7


El modelo apenas supera el 48,10% que da predecir siempre US, y el **F1 macro de 0,25** confirma que funciona mal en las clases chicas: la exactitud alta viene de acertar la clase dominante, no de distinguir países.

#### El escalado cambia todo

La consigna no lo pide, así que la respuesta del cuestionario es la de arriba. Pero vale ver qué pasa al escalar, porque es la diferencia entre un modelo inservible y uno bueno.

In [19]:
from sklearn.preprocessing import StandardScaler

scaler_el = StandardScaler().fit(X_train_el)

mlp_escalado = MLPClassifier(hidden_layer_sizes=(6, 12), max_iter=100, random_state=17)
mlp_escalado.fit(scaler_el.transform(X_train_el), y_train_el)
y_pred_esc = mlp_escalado.predict(scaler_el.transform(X_test_el))

print(f'{"":22} {"Accuracy":>10} {"F1 macro":>10}')
print(f'{"Sin escalar (consigna)":22} {accuracy_el:>10.4f} {f1_score(y_test_el, y_pred_el, average="macro"):>10.4f}')
print(f'{"Con StandardScaler":22} {accuracy_score(y_test_el, y_pred_esc):>10.4f} {f1_score(y_test_el, y_pred_esc, average="macro"):>10.4f}')

                         Accuracy   F1 macro
Sin escalar (consigna)     0.5856     0.2491
Con StandardScaler         0.9911     0.9143


/opt/anaconda3/lib/python3.12/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (100) reached and the optimization hasn't converged yet.
  warnings.warn(


De **58,56% a 99,11%** con la misma arquitectura y la misma semilla: lo único que cambia es el escalado.

La razón está en las features: `price` y `points` conviven con códigos de etiqueta de `winery` que llegan a valores de cinco cifras. Sin estandarizar, esas columnas de magnitud grande dominan la suma ponderada y las neuronas arrancan saturadas.

Es exactamente el problema que el manual describe en el capítulo 5. Para el cuestionario corresponde responder **58,56%**, que es lo que da la consigna tal como está escrita.